# P-P test — mlgw_bbh_jax (SEOBNRv5HM) + GWgpu-jax acceptance-walk NS (referee comment 1a)

**Self-contained Colab GPU notebook.** Runtime → Change runtime type → **GPU** (A100/L4/T4).

What it does: 100 injections drawn from the sampling prior (no SNR cut), each analysed with the
same pipeline as the paper's GW150914 run — mlgw_bbh_jax model_4 (2,2) mode, 4 s @ 4096 Hz,
20–1024 Hz, H1+L1, design-PSD Gaussian noise — then a P-P plot with KS / Fisher p-values.

* **Wide priors** from the surrogate training range: Mc∈[15,50] M☉, q∈[0.125,1], |χ|≤0.9,
  d_L∝d² on [50,500] Mpc, isotropic sky/orientation, tc ±0.1 s.
* **TD→FD recipe**: 0.12 s kept after the merger peak (`--mlgw-final-time`) — the whole
  ringdown up to M_tot≈200 M☉. The paper's 0.01 s truncated it (see `../waveform_check/`).
* **Sampler** = Prathaban et al. 2025 injection-study settings: nlive 1400, num_delete 700,
  n_target 60, maxmcmc 5000, stop at Δln Z < ln 0.001.
* **Resumable**: every injection is saved to Google Drive (`RUN_DIR/inj_NNN/`). After a disconnect,
  just re-run cells 1→6: completed injections are skipped. Two Colab accounts/sessions can share
  the work by using different `START/STOP` ranges on the **same** `RUN_DIR`.

**Order:** run cell 1, *restart the session* if it says so, then run everything from cell 2 down.
Run the **timing check** (cell 6 with `STOP = START + 2`) first to see the per-injection cost.

## 1. Install GWgpu-jax (+ mlgw extras) — restart the session afterwards if JAX was re-pinned

In [ ]:
import importlib.util
OWNER, REPO, BRANCH = "saulo-albuquerque-phys", "GWgpu-jax", "sharpy_sampler"
if importlib.util.find_spec("gwgpu_jax") is None:
    !pip uninstall -y -q jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt 2>/dev/null
    !pip install -q "jax[cuda12]==0.4.31" "jaxlib==0.4.31"
    !pip install -q "gwgpu_jax[data,mlgw] @ git+https://github.com/{OWNER}/{REPO}.git@{BRANCH}"
    !pip install -q corner
    print(">>> Installed. Now: Runtime -> Restart session, then run from cell 2 down.")
else:
    print("gwgpu_jax already importable - skipping install.")

## 2. mlgw_bbh_jax sub-repo + acceptance-walk kernel + Google Drive

In [ ]:
import os, subprocess, shutil
from pathlib import Path
import gwgpu_jax

# mlgw_bbh_jax is a git submodule -> not inside the pip wheel; clone it (same as the paper notebooks).
DEST = Path(gwgpu_jax.__file__).resolve().parent / "mlgw_jax" / "mlgw_bbh_jax"
if not (DEST / "mlgw" / "GW_generator.py").exists():
    if DEST.exists():
        shutil.rmtree(DEST)
    subprocess.run(["git", "clone", "--depth", "1", "-b", "gwjax_compatible",
                    "https://github.com/adrianomascioli/MLGW-JAX.git", str(DEST)], check=True)
print("mlgw_bbh_jax :", DEST)

# Acceptance-walk kernel (mrosep/blackjax_ns_gw, separately licensed, not bundled).
KDIR = "/content/blackjax_ns_gw"
if not os.path.isdir(f"{KDIR}/src/custom_kernels"):
    subprocess.run(["git", "clone", "--no-checkout", "--depth", "1", "--filter=blob:none",
                    "https://github.com/mrosep/blackjax_ns_gw.git", KDIR], check=True)
    subprocess.run(["git", "-C", KDIR, "sparse-checkout", "set", "src/custom_kernels"], check=True)
    subprocess.run(["git", "-C", KDIR, "checkout"], check=True)
os.environ["PYTHONPATH"] = f"{KDIR}/src:" + os.environ.get("PYTHONPATH", "")
os.environ["GWJAX_ACCEPTANCE_WALK_SRC"] = f"{KDIR}/src"
print("kernel       :", KDIR)

from google.colab import drive
drive.mount("/content/drive")

## 3. Write the campaign runner (`run_pp_mlgw_bbh.py`, identical to the repo copy)

In [ ]:
%%writefile /content/run_pp_mlgw_bbh.py
#!/usr/bin/env python
"""P-P injection campaign for the mlgw_bbh_jax (SEOBNRv5HM) GWgpu-jax pipeline.

Referee comment 1(a): tests the statistical calibration of the *full* pipeline
used for GW150914 in the paper — mlgw_bbh_jax TD->FD waveform, gwgpu_jax
likelihood/network projection, acceptance-walk nested sampler — on injections
drawn from the sampling prior.

Design (each choice is what makes the P-P test strictly valid):
  * injections are drawn FROM THE SAME PRIOR the sampler uses, no SNR cut
    (an SNR selection on theta while sampling the unconditioned prior biases
    the P-P curves, mostly in distance/inclination);
  * injection and recovery use the same model (mlgw_bbh_jax, (2,2) by default,
    identical to the paper's GW150914 run), so the test isolates the
    pipeline/sampler, not waveform systematics;
  * Gaussian noise coloured by the design PSD, a fresh seed per injection;
  * the signal is injected with the sampler's own tc + dt_ifo convention
    (network.project_waveform alone would leave tc unrecoverable);
  * sampler settings follow the bilby-validated acceptance-walk injection study
    (Prathaban et al. 2025): nlive 1400, num_delete 700, n_target 60,
    maxmcmc 5000, stop at logZ_live - logZ < ln(0.001).

Priors are WIDE, set by the surrogate's training range (model_4 README:
q in [1,10], |chi| <= 0.9, t_min = 2 s/M_sun) rather than by GW150914 — this also
answers referee comment 1(c). Mc >= 15 M_sun keeps the 20 Hz inspiral inside the
4 s segment (tau_N(20 Hz; Mc=15) ~= 2.4 s, + 0.4 s Tukey taper).
The volumetric distance prior is capped at 500 Mpc (brackets GW150914's 440 Mpc):
with the design PSD this gives median network SNR ~7 with a tail to strong
signals. A 2000 Mpc cap gives median SNR 1.7 (98% of injections essentially
noise) — still a valid P-P test, but one that barely exercises the masses.

Stop-and-continue: every injection writes <run-dir>/inj_NNN/{result.json,
posterior_samples.npz}; a result.json with completed=true is skipped on re-run.
Use --start/--stop to split the campaign across Colab sessions / GPUs.

    # full campaign (GPU)
    python run_pp_mlgw_bbh.py --run-dir /content/drive/MyDrive/gwjax_pp/mlgw_bbh_pp

    # half of it on a second GPU
    python run_pp_mlgw_bbh.py --run-dir ... --start 50 --stop 100

    # local smoke test (pipeline only, NOT calibrated)
    python run_pp_mlgw_bbh.py --run-dir /tmp/pp_smoke --n-inj 2 --nlive 60 \
        --num-delete 30 --n-target 5 --max-mcmc 200 --max-iters 40 --num-posterior 100

Afterwards:  python make_pp_plot.py --run-dir <run-dir>
"""
import argparse
import json
import time
from pathlib import Path

import numpy as np

ap = argparse.ArgumentParser(description=__doc__,
                             formatter_class=argparse.RawDescriptionHelpFormatter)
ap.add_argument("--run-dir", default="pp_mlgw_bbh_run")
ap.add_argument("--n-inj", type=int, default=100)
ap.add_argument("--start", type=int, default=0)
ap.add_argument("--stop", type=int, default=None)
# data segment — same as the paper's GW150914 analysis
ap.add_argument("--detectors", nargs="+", default=["H1", "L1"])
ap.add_argument("--duration", type=float, default=4.0)
ap.add_argument("--sampling-rate", type=float, default=4096.0)
ap.add_argument("--f-min", type=float, default=20.0)
ap.add_argument("--f-max", type=float, default=1024.0)
ap.add_argument("--mlgw-final-time", type=float, default=0.12,
                help="post-peak time kept on the TD grid [s]. Must cover mlgw's native end "
                     "(0.00052 s/M_sun * M_tot, i.e. 0.104 s at the prior's max M_tot~200): "
                     "the paper's 0.01 s truncates the ringdown (MM~3e-3 at M=65, near-peak at M>150)")
ap.add_argument("--modes", default="22",
                help="comma-separated mlgw modes, e.g. '22' (paper) or '22,21,33,44'")
# priors
ap.add_argument("--mc", type=float, nargs=2, default=(15.0, 50.0))
ap.add_argument("--q", type=float, nargs=2, default=(0.125, 1.0))
ap.add_argument("--chi", type=float, nargs=2, default=(-0.9, 0.9))
ap.add_argument("--distance", type=float, nargs=2, default=(50.0, 500.0))
ap.add_argument("--tc-halfwidth", type=float, default=0.1)
# sampler (Prathaban et al. 2025 injection-study settings)
ap.add_argument("--nlive", type=int, default=1400)
ap.add_argument("--num-delete", type=int, default=700)
ap.add_argument("--n-target", type=int, default=60)
ap.add_argument("--max-mcmc", type=int, default=5000)
ap.add_argument("--max-iters", type=int, default=20000)
ap.add_argument("--log-dlogz", type=float, default=float(np.log(1e-3)))
ap.add_argument("--num-posterior", type=int, default=4000)
# seeds
ap.add_argument("--inj-seed", type=int, default=20261104)
ap.add_argument("--noise-seed0", type=int, default=1000)
ap.add_argument("--sampler-seed0", type=int, default=7000)
ap.add_argument("--force", action="store_true", help="re-run completed injections")
args, _ = ap.parse_known_args()   # tolerate Jupyter's -f flag

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

import gwgpu_jax
from gwgpu_jax.mlgw_jax.mlgw_bbh_jax_waveform_generator import MLGWBBHGenerator

RUN_DIR = Path(args.run_dir)
RUN_DIR.mkdir(parents=True, exist_ok=True)
MODES = [tuple(int(c) for c in m.strip()) for m in args.modes.split(",")]
MODES_ARG = MODES[0] if len(MODES) == 1 else MODES

# ── grid / waveform ──────────────────────────────────────────────────────────
grid = gwgpu_jax.TimeFrequencyGrid(duration=args.duration, sampling_rate=args.sampling_rate,
                                   f_min=args.f_min, f_max=args.f_max)
FREQS = grid.frequency_domain_array

# TD->FD recipe of the paper notebook / bbh_case runner (zero-crossing time grid +
# asymmetric Tukey taper), but with a post-peak span long enough for the whole
# ringdown. mlgw returns exact zeros past its native end, so a long span is safe.
# NB: the model's merger sits at (tc - MLGW_FINAL_TIME) mod duration.
MLGW_FINAL_TIME = args.mlgw_final_time
TUKEY_ALPHA_LEFT, TUKEY_ALPHA_RIGHT = 0.2, 0.001


def _tukey_asymmetric(n, a_l, a_r):
    x = jnp.linspace(0.0, 1.0, n)
    w = jnp.ones_like(x)
    left = 0.5 * (1.0 + jnp.cos(2.0 * jnp.pi * (x / a_l - 0.5)))
    right = 0.5 * (1.0 + jnp.cos(2.0 * jnp.pi * ((x - 1.0) / a_r + 0.5)))
    w = jnp.where(x < a_l / 2.0, left, w)
    return jnp.where(x > 1.0 - a_r / 2.0, right, w)


_bbh = MLGWBBHGenerator(model=4)
_gen = _bbh.generate
_dt, _N = grid.dt, grid.n_samples
_T_GRID = jnp.arange(_N) * _dt - grid.duration + MLGW_FINAL_TIME
_WINDOW = _tukey_asymmetric(_N, TUKEY_ALPHA_LEFT, TUKEY_ALPHA_RIGHT)


def mc_q_to_m1_m2(mc, q):
    m1 = mc * (1.0 + q) ** 0.2 / q ** 0.6
    return m1, q * m1


def waveform_fn(params, freqs):
    m1, m2 = mc_q_to_m1_m2(params["mc"], params["q"])
    theta = jnp.stack([m1, m2, params["chi_1"], params["chi_2"],
                       params["distance"], params["inclination"], params["phi_c"]])
    hp_td, hc_td = _gen(theta, _T_GRID, modes=MODES_ARG)
    return (jnp.fft.rfft(hp_td * _WINDOW) * _dt,
            -jnp.fft.rfft(hc_td * _WINDOW) * _dt)


# ── priors (used to draw injections AND inside the sampler) ─────────────────
TC_CENTER = 0.875 * args.duration       # paper convention (3.5 s of a 4 s segment)
PI = float(np.pi)
PARAM_BOUNDS = {
    "mc":          tuple(args.mc),
    "q":           tuple(args.q),
    "chi_1":       tuple(args.chi),
    "chi_2":       tuple(args.chi),
    "distance":    tuple(args.distance),
    "inclination": (0.0, PI),
    "ra":          (0.0, 2 * PI),
    "dec":         (-PI / 2, PI / 2),
    "psi":         (0.0, PI),
    "phi_c":       (0.0, 2 * PI),
    "tc":          (TC_CENTER - args.tc_halfwidth, TC_CENTER + args.tc_halfwidth),
}
PRIORS = {"inclination": "sin", "dec": "cos", "distance": "volumetric"}
NAMES = list(PARAM_BOUNDS)

cfg = dict(vars(args), modes=args.modes, param_bounds=PARAM_BOUNDS, priors=PRIORS,
           waveform="mlgw_bbh_jax model_4 (SEOBNRv5HM)", tc_center=TC_CENTER,
           mlgw_final_time=MLGW_FINAL_TIME,
           tukey=(TUKEY_ALPHA_LEFT, TUKEY_ALPHA_RIGHT),
           gwgpu_jax_version=getattr(gwgpu_jax, "__version__", "?"))
cfg_file = RUN_DIR / "config.json"
if cfg_file.exists():
    old = json.load(open(cfg_file))
    for k in ("param_bounds", "priors", "inj_seed", "modes", "duration", "f_min", "f_max",
              "mlgw_final_time"):
        if json.dumps(old.get(k)) != json.dumps(cfg.get(k)):
            raise SystemExit(f"config mismatch on {k!r} vs existing {cfg_file}: "
                             f"{old.get(k)} != {cfg.get(k)} — use a new --run-dir")
else:
    json.dump(cfg, open(cfg_file, "w"), indent=2, default=list)

# ── injections (cached so truths persist across restarts) ───────────────────
INJ_FILE = RUN_DIR / "injections.json"
if INJ_FILE.exists():
    injections = json.load(open(INJ_FILE))
else:
    specs = gwgpu_jax.resolve_priors(PARAM_BOUNDS, PRIORS)
    draw = gwgpu_jax.sample_prior(jax.random.PRNGKey(args.inj_seed), args.n_inj, specs)[0]
    injections = [{n: float(np.asarray(draw[n])[i]) for n in NAMES} for i in range(args.n_inj)]
    json.dump(injections, open(INJ_FILE, "w"), indent=2)
print(f"{len(injections)} injections in {INJ_FILE}")


def run_one(i, truth):
    inj_dir = RUN_DIR / f"inj_{i:03d}"
    res_file = inj_dir / "result.json"
    if res_file.exists() and not args.force and json.load(open(res_file)).get("completed"):
        return None
    inj_dir.mkdir(exist_ok=True)

    network = gwgpu_jax.Network.from_names(args.detectors, grid)
    network.generate_noise(seed=args.noise_seed0 + i)
    hp, hc = waveform_fn({k: jnp.asarray(v) for k, v in truth.items()}, FREQS)
    gmst = float(network.gmst)
    h = {}
    for ifo in network.interferometers:
        Fp, Fc = ifo.antenna_pattern(truth["ra"], truth["dec"], truth["psi"], gmst)
        dt = ifo.time_delay_from_geocenter(truth["ra"], truth["dec"], gmst)
        h[ifo.name] = gwgpu_jax.waveform_projection_fd(hp, hc, Fp, Fc, FREQS, truth["tc"] + dt)
    snr = float(network.network_optimal_snr(h))
    network.inject_signal(h, domain="fd")

    sampler = gwgpu_jax.GWgpu_jaxAcceptanceWalkSampler(
        network=network, waveform_fn=waveform_fn, param_bounds=PARAM_BOUNDS,
        priors=PRIORS, gmst=None, periodic_params=("phi_c", "ra", "psi"))
    t0 = time.perf_counter()
    res = sampler.run(
        rng_key=jax.random.PRNGKey(args.sampler_seed0 + i),
        num_live=args.nlive, n_target=args.n_target, max_mcmc=args.max_mcmc,
        num_delete=args.num_delete, max_iterations=args.max_iters,
        log_dlogz_target=args.log_dlogz, num_posterior_samples=args.num_posterior,
        verbose=False)
    elapsed = time.perf_counter() - t0

    post = {n: np.asarray(res.posterior_samples[n]) for n in NAMES}
    np.savez_compressed(inj_dir / "posterior_samples.npz", **post)
    rec = dict(index=i, completed=True, truth=truth, network_snr=snr,
               p_values={n: float(np.mean(post[n] < truth[n])) for n in NAMES},
               logZ=float(res.logZ), logZ_err=float(res.logZ_err), ess=float(res.ess),
               n_iterations=int(res.n_iterations), elapsed_s=elapsed)
    json.dump(rec, open(res_file, "w"), indent=2)
    return rec


stop = len(injections) if args.stop is None else min(args.stop, len(injections))
for i in range(args.start, stop):
    rec = run_one(i, injections[i])
    if rec is None:
        print(f"[{i:03d}] already complete — skipped")
        continue
    print(f"[{i:03d}] SNR={rec['network_snr']:5.1f}  ESS={rec['ess']:6.0f}  "
          f"iters={rec['n_iterations']:4d}  logZ={rec['logZ']:+9.2f}  "
          f"{rec['elapsed_s']:6.0f} s", flush=True)
print("done. Plot with:  python make_pp_plot.py --run-dir", RUN_DIR)


## 4. Write the P-P plotter (`make_pp_plot.py`)

In [ ]:
%%writefile /content/make_pp_plot.py
#!/usr/bin/env python
"""P-P plot + calibration statistics from a P-P campaign directory.

Works on any run dir whose inj_NNN/result.json files hold `truth` and
`p_values` (run_pp_mlgw_bbh.py, and the earlier ripplegw P-P notebooks).

Reports, as in the LVK/bilby P-P convention (e.g. Romero-Shaw et al. 2020):
  * empirical CDF of the credible level of the truth, per parameter,
    with 1/2/3-sigma binomial bands;
  * per-parameter KS p-value against U(0,1) and Fisher-combined p-value;
  * a derived chi_eff curve, recomputed from the stored posteriors
    (shows the best-measured spin combination is calibrated too).

    python make_pp_plot.py --run-dir pp_mlgw_bbh_run [--out pp_plot.pdf]
"""
import argparse
import glob
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

ap = argparse.ArgumentParser(description=__doc__,
                             formatter_class=argparse.RawDescriptionHelpFormatter)
ap.add_argument("--run-dir", required=True)
ap.add_argument("--out", default=None, help="default: <run-dir>/pp_plot.pdf")
ap.add_argument("--no-chi-eff", action="store_true")
args = ap.parse_args()

run = Path(args.run_dir)
recs = [json.load(open(f)) for f in sorted(glob.glob(str(run / "inj_*" / "result.json")))]
recs = [r for r in recs if r.get("completed")]
if len(recs) < 2:
    raise SystemExit(f"only {len(recs)} completed injections in {run}")
names = list(recs[0]["p_values"])

LABELS = {"mc": r"$\mathcal{M}_c$", "Mc": r"$\mathcal{M}_c$", "q": r"$q$",
          "chi_1": r"$\chi_1$", "chi_2": r"$\chi_2$", "distance": r"$d_L$",
          "inclination": r"$\theta_{JN}$", "ra": r"$\alpha$", "dec": r"$\delta$",
          "psi": r"$\psi$", "phi_c": r"$\phi_c$", "tc": r"$t_c$",
          "chi_eff": r"$\chi_{\rm eff}$"}

pvals = {n: np.array([r["p_values"][n] for r in recs]) for n in names}

# derived chi_eff = (m1 chi1 + m2 chi2)/(m1+m2) = (chi1 + q chi2)/(1+q), q = m2/m1
qn = "q" if "q" in names else None
if qn and "chi_1" in names and not args.no_chi_eff:
    pe = []
    for r in recs:
        f = run / f"inj_{r['index']:03d}" / "posterior_samples.npz"
        if not f.exists():
            pe = None
            break
        p = np.load(f)
        ce = (p["chi_1"] + p[qn] * p["chi_2"]) / (1 + p[qn])
        t = r["truth"]
        pe.append(np.mean(ce < (t["chi_1"] + t[qn] * t["chi_2"]) / (1 + t[qn])))
    if pe is not None:
        pvals["chi_eff"] = np.array(pe)

N = len(recs)
x = np.linspace(0, 1, 1001)
fig, ax = plt.subplots(figsize=(6.5, 6.5))
for sig, a in [(1, 0.25), (2, 0.15), (3, 0.08)]:
    e = stats.norm.cdf(sig)
    ax.fill_between(x, stats.binom.ppf(1 - e, N, x) / N, stats.binom.ppf(e, N, x) / N,
                    color="0.4", alpha=a, lw=0)
ax.plot([0, 1], [0, 1], "k--", lw=0.8)

ks = {}
cmap = plt.get_cmap("tab20")
for j, (n, p) in enumerate(pvals.items()):
    ks[n] = float(stats.kstest(p, "uniform").pvalue)
    ax.plot(x, np.searchsorted(np.sort(p), x, side="right") / N, lw=1.4,
            color=cmap(j % 20), label=f"{LABELS.get(n, n)} ({ks[n]:.2f})")
comb = float(stats.combine_pvalues([ks[n] for n in names], method="fisher").pvalue)

ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
ax.set_xlabel("C.I."); ax.set_ylabel("Fraction of events in C.I.")
ax.set_title(f"N = {N}, combined p-value = {comb:.3f}")
ax.legend(fontsize=8, loc="upper left", title="KS p-value", title_fontsize=8)
out = Path(args.out) if args.out else run / "pp_plot.pdf"
fig.savefig(out, bbox_inches="tight")
fig.savefig(out.with_suffix(".png"), dpi=150, bbox_inches="tight")

snr = np.array([r["network_snr"] for r in recs])
el = np.array([r["elapsed_s"] for r in recs])
summary = dict(n=N, ks_pvalues=ks, combined_pvalue_fisher=comb,
               combined_over=names,
               snr_min=float(snr.min()), snr_median=float(np.median(snr)),
               snr_max=float(snr.max()), n_snr_ge_8=int((snr >= 8).sum()),
               wall_time_median_s=float(np.median(el)), wall_time_total_h=float(el.sum() / 3600))
json.dump(summary, open(run / "pp_summary.json", "w"), indent=2)

print(f"N = {N}   SNR {snr.min():.1f}–{snr.max():.1f} (median {np.median(snr):.1f}, "
      f"{(snr >= 8).sum()} with SNR>=8)")
for n, p in ks.items():
    print(f"  {n:12s} KS p = {p:.3f}")
print(f"combined (Fisher, sampled params) p = {comb:.3f}")
print(f"wall time per injection: median {np.median(el)/60:.1f} min, total {el.sum()/3600:.1f} h")
print("saved", out, "and", run / "pp_summary.json")


## 5. Configuration

Keep `RUN_DIR` fixed for the whole campaign (the runner refuses to mix configurations in one
folder). `EXTRA` passes any CLI override, e.g. `"--sampling-rate 2048"` to halve the waveform
cost (keeps the 1024 Hz upper cut-off) — but then use a **new** `RUN_DIR`.

In [ ]:
RUN_DIR = "/content/drive/MyDrive/gwjax_referee/pp_mlgw_bbh"
START, STOP = 0, 2        # timing check first (2 injections); then set STOP = 100
EXTRA = ""                # e.g. "--sampling-rate 2048"

import jax
print("JAX devices:", jax.devices())
assert jax.devices()[0].platform == "gpu", "No GPU - Runtime -> Change runtime type -> GPU"

## 6. Run (resumable — one line per injection: SNR, ESS, NS iterations, logZ, wall time)

In [ ]:
!cd /content && python -u run_pp_mlgw_bbh.py --run-dir "{RUN_DIR}" --start {START} --stop {STOP} {EXTRA} 2>&1 | grep -vE "features_spec|ListWrapper|keras|^ *$|oneDNN|cuda_|TF-TRT|tensorflow"

## 7. P-P plot + KS / Fisher p-values (works on a partial campaign)

In [ ]:
!cd /content && python make_pp_plot.py --run-dir "{RUN_DIR}"
from IPython.display import Image, display
display(Image(f"{RUN_DIR}/pp_plot.png"))

## 8. (optional) Zip the results for download — posteriors + result.json + plots

In [ ]:
import shutil
z = shutil.make_archive("/content/pp_mlgw_bbh_results", "zip", RUN_DIR)
from google.colab import files
files.download(z)